The optical CNOT gate implements a conditional-NOT operation on dual-rail single-photon qubits using a 6-mode linear optical network with coincidence post-selection. The structure relies on two spatial modes each for the control and target qubits, alongside two auxiliary (ancilla) optical modes that mediate effective photon-photon interactions through quantum interference. Because linear optical components alone cannot deterministically perform non-linear gates, the circuit operates probabilistically, yielding the desired transformed state upon measuring two output photons in a coincidence post-selection detection setup with a maximum theoretical success probability of $1/9 \approx 11.1\%$.

In [1]:
import numpy as np
import strawberryfields as sf
from strawberryfields.ops import Fock, BSgate, Rgate

In [2]:
THETA_13 = np.arccos(1/np.sqrt(3))
THETA_PI4 = np.pi/4               

In [3]:
params = (THETA_13, 0.0, 
          THETA_PI4, 0.0, 
          THETA_13, 0.0, 
          THETA_13, 0.0, 
          THETA_PI4, np.pi)
def ralph_cnot(input_state, theta1, phi1, theta2, phi2, theta3, phi3, theta4, phi4, theta5, phi5, cutoff_dim=3):
    prog = sf.Program(6)
    with prog.context as q:
        for mode, photons in enumerate(input_state):
            if photons > 0:
                Fock(photons) | q[mode]
        BSgate(theta1, phi1) | (q[0], q[1])
        BSgate(theta2, phi2) | (q[3], q[4])
        BSgate(theta3, phi3) | (q[2], q[3])
        BSgate(theta4, phi4) | (q[4], q[5])
        BSgate(theta5, phi5) | (q[3], q[4])
    eng = sf.Engine("fock", backend_options={"cutoff_dim": cutoff_dim})
    return eng.run(prog)
    
inputs = {
    "|00>": [0, 1, 0, 1, 0, 0],
    "|01>": [0, 1, 0, 0, 1, 0],
    "|10>": [0, 0, 1, 1, 0, 0],
    "|11>": [0, 0, 1, 0, 1, 0]
}

desired_outputs = {
    "|00>": (0, 1, 0, 1, 0, 0),
    "|01>": (0, 1, 0, 0, 1, 0),
    "|10>": (0, 0, 1, 0, 1, 0),
    "|11>": (0, 0, 1, 1, 0, 0)
}

for name, input_state in inputs.items():
    result = ralph_cnot(input_state, *params)
    probs = result.state.all_fock_probs()
    print("\n")
    print("Input:", name)
    desired_output = desired_outputs[name]
    success_probability = probs[desired_output]
    print("Desired output:", desired_output)
    print("Success probability:", success_probability)
    for outcome, probability in np.ndenumerate(probs):
        if probability > 1e-8:
            print(outcome, "probability =", probability)

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\strawberryfields\program.py:732: UserWarning: The circuit consists of 2 disconnected components.




Input: |00>
Desired output: (0, 1, 0, 1, 0, 0)
Success probability: 0.11111111111111122
(0, 1, 0, 0, 0, 1) probability = 0.11111111111111116
(0, 1, 0, 1, 0, 0) probability = 0.11111111111111122
(0, 1, 1, 0, 0, 0) probability = 0.11111111111111116
(1, 0, 0, 0, 0, 1) probability = 0.22222222222222227
(1, 0, 0, 1, 0, 0) probability = 0.22222222222222235
(1, 0, 1, 0, 0, 0) probability = 0.22222222222222227


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\strawberryfields\program.py:732: UserWarning: The circuit consists of 2 disconnected components.




Input: |01>
Desired output: (0, 1, 0, 0, 1, 0)
Success probability: 0.11111111111111122
(0, 1, 0, 0, 0, 1) probability = 0.11111111111111116
(0, 1, 0, 0, 1, 0) probability = 0.11111111111111122
(0, 1, 1, 0, 0, 0) probability = 0.11111111111111116
(1, 0, 0, 0, 0, 1) probability = 0.22222222222222227
(1, 0, 0, 0, 1, 0) probability = 0.22222222222222235
(1, 0, 1, 0, 0, 0) probability = 0.22222222222222227


Input: |10>
Desired output: (0, 0, 1, 0, 1, 0)
Success probability: 0.11111111111111113
(0, 0, 0, 0, 1, 1) probability = 0.11111111111111115
(0, 0, 0, 1, 0, 1) probability = 0.11111111111111115
(0, 0, 0, 1, 1, 0) probability = 0.11111111111111117
(0, 0, 0, 2, 0, 0) probability = 0.22222222222222243
(0, 0, 1, 0, 0, 1) probability = 0.11111111111111116
(0, 0, 1, 0, 1, 0) probability = 0.11111111111111113
(0, 0, 2, 0, 0, 0) probability = 0.22222222222222238


Input: |11>
Desired output: (0, 0, 1, 1, 0, 0)
Success probability: 0.11111111111111113
(0, 0, 0, 0, 1, 1) probability = 0.111111